# EUMETSAT Data Tailor: `tailor=` (config + eligibility, no network)

EUMETSAT **Data Tailor** customises products *server-side* — subset (ROI crop), reproject, and reformat — before delivery. earthlens reaches it through a dedicated `tailor=TailorConfig(...)` argument to `EUMETSAT.download`.

This notebook teaches the **request shape** and **eligibility model** entirely offline: how to build a `TailorConfig`, how its `bbox` becomes a Data Tailor region-of-interest, and which catalog collections are Data-Tailor-eligible. The **live** submit → poll → stream → delete round-trip needs Data Store download entitlement and is only described here (last section).

API: `earthlens.eumetsat.TailorConfig`, `earthlens.eumetsat.Catalog`.

## Setup

Import `TailorConfig` and `Catalog` from the EUMETSAT backend, plus `pandas` / `matplotlib` for the tables and chart. No credentials or network are needed — both ship as package data / pure value objects.

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd

from earthlens.eumetsat import Catalog, TailorConfig

## Quickstart — a `TailorConfig`

A `TailorConfig` is a small frozen value object describing the customisation: the output `format`, the target `crs` / projection, an optional `bbox` crop, an optional band `filter`, and a `quicklook` flag. The Data Tailor product-type is **not** set here — it comes from the catalog row (next sections).

In [ ]:
cfg = TailorConfig(format="geotiff", crs="geographic", bbox=(4.0, 48.0, 8.0, 52.0))
cfg

The fields map onto the `eumdac` `Chain` the backend builds:

| Field | Meaning | Maps to |
|-------|---------|---------|
| `format` | output format (`"geotiff"`, `"netcdf4"`, …) | `Chain.format` |
| `crs` | target projection, or `None` for none | `Chain.projection` |
| `bbox` | `(west, south, east, north)` crop | `Chain.roi` (`NSWE`) |
| `filter` | band / layer names to keep | `Chain.filter` |
| `quicklook` | request a quicklook rendering | `Chain.quicklook` |

### Native output formats need `crs=None`

A handful of Data Tailor output formats carry their own fixed grid and reject any projection: `"msgnative"`, `"epsnative"`, `"hrit"`, `"hrit_compressed"`. `TailorConfig` requires `crs=None` for these -- it maps to leaving `Chain.projection` unset -- and rejects the combination of a native `format` with any other `crs` at construction time, before a request ever reaches Data Tailor.

In [ ]:
native_cfg = TailorConfig(format="msgnative", crs=None, bbox=(-5.0, 40.0, 15.0, 55.0))
native_cfg

## ROI mapping — `bbox` → `NSWE`

Data Tailor's region-of-interest is an `[north, south, west, east]` list. `TailorConfig` stores `bbox` in the GeoJSON/OGC order `(west, south, east, north)` and exposes the reordered ROI via `.nswe`. When no `bbox` is given, `.nswe` is `None` and the backend falls back to the request's own `lat_lim` / `lon_lim`.

In [ ]:
examples = [
    ("Central Europe", (4.0, 48.0, 8.0, 52.0)),
    ("British Isles", (-11.0, 49.0, 2.0, 61.0)),
    ("no bbox (falls back to lat/lon_lim)", None),
]
pd.DataFrame(
    [(name, bbox, TailorConfig(bbox=bbox).nswe) for name, bbox in examples],
    columns=["region", "bbox (w, s, e, n)", ".nswe [n, s, w, e]"],
)

## Validation — bad boxes are rejected early

`TailorConfig` validates `bbox` at construction: an inverted box (`west > east` or `south > north`) or an out-of-range coordinate raises a `ValidationError` at the call site, rather than failing far away at the live service.

`TailorConfig` validates `bbox` at construction: an inverted box (`west > east` or `south > north`) or an out-of-range coordinate raises a `pydantic.ValidationError` at the call site — e.g. `TailorConfig(bbox=(8, 48, 4, 52))` fails with *"bbox west (8) must be <= east (4)"* — rather than failing far away at the live service.

## Eligibility — not every collection is tailorable

Data Tailor supports a fixed registry of product types. A curated catalog row is Data-Tailor-eligible only when it carries a `tailor_product_type`. Load the catalog and split the collections into eligible vs not.

In [ ]:
cat = Catalog()
eligible = {k: c for k, c in cat.datasets.items() if c.tailor_product_type}
print(
    f"{len(eligible)} of {len(cat.datasets)} curated collections are Data-Tailor-eligible"
)

### Eligible collections by group

Tally the eligible collections by mission-family `group` and plot it, so the spread across MSG / MTG / Metop / Sentinel-3 / … is visible at a glance.

In [ ]:
by_group = Counter(c.group.value for c in eligible.values())
series = pd.Series(dict(sorted(by_group.items()))).sort_values()

ax = series.plot.barh(color="steelblue")
ax.set_xlabel("eligible collections")
ax.set_title("Data-Tailor-eligible EUMETSAT collections by group")
plt.tight_layout()
plt.show()

### A sample of eligible rows and their product type

Each eligible row maps its friendly key to a Data Tailor `tailor_product_type` (the id passed as `Chain.product`). Show a representative sample.

In [ ]:
sample = sorted(eligible.items())[:12]
pd.DataFrame(
    [(k, c.group.value, c.output_kind, c.tailor_product_type) for k, c in sample],
    columns=["key", "group", "output_kind", "tailor_product_type"],
)

A `tailor=` request against a **non-eligible** collection (e.g. Sentinel-5P TROPOMI, which this service does not tailor) raises a clear `ValueError` before any job is submitted — confirm one is `None`:

In [ ]:
print(
    "s5p-l2-no2 tailor_product_type:", cat.get_dataset("s5p-l2-no2").tailor_product_type
)

## The live round-trip (requires download entitlement)

With valid credentials **and** Data Store download entitlement for the collection, a `tailor=` download submits one customisation per matching product, polls it to `DONE`, streams the customised output(s) to `path`, and deletes the customisation (quota hygiene). It returns the customised GeoTIFF/NetCDF paths, which pyramids opens like any raster:

```python
from earthlens.core import EarthLens
from earthlens.eumetsat import TailorConfig

el = EarthLens(
    data_source="eumetsat",
    start="2024-06-01", end="2024-06-01",
    variables={"s3-olci-l1-efr": ["OLL1EFR"]},
    lat_lim=[50.0, 52.0], lon_lim=[-1.0, 1.0],
    path="eumetsat_tailor_out",
)
paths = el.download(tailor=TailorConfig(format="geotiff", crs="geographic", bbox=(-1.0, 50.0, 1.0, 52.0)))
```

This cell is intentionally **not executed** here: it needs an account authorised to download from the Data Store (accept the collection licence in the [EUMETSAT portal](https://user.eumetsat.int)). Everything above runs offline.

## Takeaway

- `TailorConfig(format=, crs=, bbox=, filter=, quicklook=)` is the spatial customisation request; `bbox` is `(west, south, east, north)` and becomes the `[n, s, w, e]` ROI via `.nswe`.
- Only catalog rows with a `tailor_product_type` are Data-Tailor-eligible; a non-eligible request fails fast with a clear error.
- `tailor=` is **spatial** and distinct from the temporal `aggregate=` reducer. See the [Data Tailor reference](../../reference/eumetsat/data-tailor.md).